# 19 卷积神经网络 CNN

依赖安装说明：`pip install numpy matplotlib scikit-learn torch`

CNN 擅长处理图像、网格和局部结构。卷积核会在图像上滑动，检测局部模式，例如边缘、纹理、形状。


## 1. 数学逻辑

二维卷积可以简化理解为：

$$Y_{i,j}=\sum_{u,v}K_{u,v}X_{i+u,j+v}$$

同一个卷积核在所有位置共享参数，所以 CNN 比全连接网络更适合图像：

- 局部连接：只看附近像素。
- 参数共享：同一个特征检测器到处使用。
- 平移等变：图案移动后仍能被类似方式检测。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 构造 8x8 小图：类别 0 是竖线，类别 1 是横线
n = 300
images = np.zeros((n, 8, 8), dtype=np.float32)
labels = np.zeros(n, dtype=np.int64)
for i in range(n):
    cls = i % 2
    labels[i] = cls
    img = np.random.normal(scale=0.15, size=(8, 8))
    if cls == 0:
        img[:, 3:5] += 1.0
    else:
        img[3:5, :] += 1.0
    images[i] = img

plt.subplot(1, 2, 1); plt.imshow(images[0], cmap='gray'); plt.title('竖线类')
plt.subplot(1, 2, 2); plt.imshow(images[1], cmap='gray'); plt.title('横线类')
plt.show()


In [ ]:
# 从零演示：一个 3x3 卷积核如何在图像上滑动

def conv2d_single(image, kernel):
    kh, kw = kernel.shape
    out = np.zeros((image.shape[0] - kh + 1, image.shape[1] - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            patch = image[i:i+kh, j:j+kw]
            out[i, j] = np.sum(patch * kernel)
    return out

vertical_kernel = np.array([[-1, 1, -1], [-1, 1, -1], [-1, 1, -1]])
feature_map = conv2d_single(images[0], vertical_kernel)
plt.imshow(feature_map, cmap='viridis')
plt.title('竖线卷积核的响应')
plt.colorbar()
plt.show()


In [ ]:
# PyTorch 实战：小 CNN 分类竖线/横线
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(images, labels, random_state=42, stratify=labels)
Xtr = torch.tensor(X_train[:, None, :, :], dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test[:, None, :, :], dtype=torch.float32)

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(8 * 4 * 4, 2),
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for step in range(120):
    logits = model(Xtr)
    loss = loss_fn(logits, ytr)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = model(Xte).argmax(dim=1).numpy()
print('CNN accuracy:', round(accuracy_score(y_test, pred), 3))


## 2. 常见误区

- CNN 不只适用于图片，但要求数据有局部结构。
- 池化会降低空间分辨率，不能随便堆太多。
- 小数据上 CNN 也会过拟合，常需要数据增强。

## 3. 小实验

- 改卷积核数量 `8`。
- 去掉 MaxPool，观察参数量和效果。
- 增大噪声，看 CNN 何时出错。
